In [1]:
import ollama
import requests
import json

In [2]:
model = "llama3.2:1b"

#### Task 1: Interact with deployed LLM via python 


**Objective:**

Explore different techniques to interact with the deployed LLM.

**Task Description:**

1. Use Request libaray (HTTP Client) and send a POST request to interact with the LLM: [How To](https://requests.readthedocs.io/en/latest/user/quickstart/#make-a-request)

In [3]:
# Simple HTTP Request via requests

# Define the URL of the deployed LLM ( this port is forwarded from the docker container to the host system)
url = "http://localhost:11434/api/generate"

# Define the prompt as json
body_json = {
    "model": model,
    "prompt": "Describe Generative AI in two sentences."
}

# ADD HERE YOUR CODE
# HINT: Send the POST request using the json body
# Sende den POST-Request an die Ollama API
response = requests.post(url, json=body_json)

# Check if the request was successful
if response.status_code == 200:
    # Process the response
    response_text = response.text

    # Convert each line to json
    response_lines = response_text.splitlines()
    response_json = [json.loads(line) for line in response_lines]
    for line in response_json:
        # Print the response. No line break
        print(line["response"], end="")
else:
    print("Error:", response.status_code)


Generative AI refers to a class of machine learning algorithms that can create new, original content, such as text, images, or music, by generating variations of existing patterns and structures, often using neural networks and other computational techniques. Unlike traditional data-driven approaches, generative AI systems can learn from large datasets and generate diverse outputs without being explicitly programmed or supervised, enabling a wide range of applications in areas like art, entertainment, and product design.

**Task Description:**

2. Use Ollama python library to interact with the LLM: [How To](https://pypi.org/project/ollama/)

- First use method ``ollama.chat(...)``
- First use method ``ollama.chat(...)`` with ``stream=True``

In [4]:
# API Call via ollama

# ADD HERE YOUR CODE

# Einfacher Chat-Aufruf
response = ollama.chat(model=model, messages=[
    {'role': 'user', 'content': 'Tell me a short joke.'}
])


print(response["message"]["content"])

A man walked into a library and asked the librarian, "Do you have any books on Pavlov's dogs and Schrödinger's cat?" The librarian replied, "It rings a bell, but I'm not sure if it's here or not."


In [5]:
# Streaming API Call via ollama

# Response streaming can be enabled by setting stream=True, 
# modifying function calls to return a Python generator where each part is an object in the stream.

# ADD HERE YOUR CODE

# Streaming Chat-Aufruf
stream = ollama.chat(
    model=model,
    messages=[{'role': 'user', 'content': 'Write a short poem about coding.'}],
    stream=True,
)

for chunk in stream:
  print(chunk["message"]["content"], end="", flush=True)

Here's a short poem about coding:

Bytes and lines, a digital space
I weave my code, with precision and pace
Characters dance, on screen so bright
As I bring to life, the world in sight

With every line, a story unfolds
A tale of code, where logic holds
The world is mine, in this digital realm
Where algorithms reign, and innovation's helm.

#### Task 2: Experimenting with Prompt Techniques

**Objective:**

Objective: Explore different prompt techniques (Zero Shot, One Shot, and Few Shot) by sending different types of prompts to the LLM.

![image](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*QSpK--jqPiUU_OHuZvtUWA.png)

**Task Description:**

1. Create three prompts for a sentiment analysis task: a Zero Shot prompt, a One Shot prompt, and a Few Shot prompt. Use the examples from the table above.
2. Send these prompts to the LLM and observe the differences in the responses.
3. Compare and discuss the responses.

In [6]:
# ADD HERE YOUR PROMPTS

zero_shot_prompt = """Entscheide, ob die folgende Bewertung positiv oder negativ ist.
Bewertung: Das Essen war absolut fantastisch!
Antwort:"""

one_shot_prompt = """Entscheide, ob die Bewertung positiv oder negativ ist.

Beispiel:
Bewertung: Der Service war schrecklich.
Antwort: Negativ

Bewertung: Der Film war sehr unterhaltsam!
Antwort:"""

few_shot_prompt = """Entscheide, ob die Bewertung positiv oder negativ ist.

Bewertung: Das Wetter ist toll. -> Positiv
Bewertung: Ich habe schlecht geschlafen. -> Negativ
Bewertung: Die Musik ist viel zu laut und nervig. -> Negativ
Bewertung: Ich liebe diesen neuen Laptop! -> Positiv

Bewertung: Der Kaffee schmeckt verbrannt.
Antwort:"""

# Stream the responses and print them
for idx, prompt in enumerate([zero_shot_prompt, one_shot_prompt, few_shot_prompt]):
    prompt_type = ["Zero-Shot", "One-Shot", "Few-Shot"][idx]
    print(f"\n--- {prompt_type} Prompt ---\n")
    print(f"User Prompt:\n{prompt}\n")
    
    stream = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    
    print("Model Output:")
    for chunk in stream:
        print(chunk["message"]["content"], end="", flush=True)
    print("\n-----------------------------\n")



--- Zero-Shot Prompt ---

User Prompt:
Entscheide, ob die folgende Bewertung positiv oder negativ ist.
Bewertung: Das Essen war absolut fantastisch!
Antwort:

Model Output:
Die Bewertung "Das Essen war absolut fantastisch!" ist positiv.
-----------------------------


--- One-Shot Prompt ---

User Prompt:
Entscheide, ob die Bewertung positiv oder negativ ist.

Beispiel:
Bewertung: Der Service war schrecklich.
Antwort: Negativ

Bewertung: Der Film war sehr unterhaltsam!
Antwort:

Model Output:
Bewertung: Der Film war sehr unterhaltsam!
-----------------------------


--- Few-Shot Prompt ---

User Prompt:
Entscheide, ob die Bewertung positiv oder negativ ist.

Bewertung: Das Wetter ist toll. -> Positiv
Bewertung: Ich habe schlecht geschlafen. -> Negativ
Bewertung: Die Musik ist viel zu laut und nervig. -> Negativ
Bewertung: Ich liebe diesen neuen Laptop! -> Positiv

Bewertung: Der Kaffee schmeckt verbrannt.
Antwort:

Model Output:
Die Bewertungen sind im Allgemeinen negativ, da sie sowo

#### Task 3: Prompt Refinement and Optimization

**Objective:** 

Refine a prompt to improve the clarity and quality of the LLM's response.

**Task Description:**

- Start with a basic prompt asking the LLM to summarize a paragraph.
- Refine the prompt by adding specific instructions to improve the summary's quality. (Example: define how long the summary should be, define on which to focus in the summary)

In [7]:
# Original prompt
original_prompt = "Summarize the following paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries."

# ADD HERE YOUR PROMPT
# Ein verfeinerter Prompt mit klaren Anweisungen
refined_prompt = """Fasse den folgenden Absatz in genau einem Satz zusammen. 
Konzentriere dich dabei nur auf die Anwendungen in der Industrie:

Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries."""

# Stream the responses and print them
for idx, prompt in enumerate([original_prompt, refined_prompt]):
    prompt_type = ["Original Prompt", "Refined Prompt"][idx]
    print(f"\n--- {prompt_type} ---\n")
    print(f"User Prompt:\n{prompt}\n")
    
    stream = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    
    print("Model Output:")
    for chunk in stream:
        print(chunk["message"]["content"], end="", flush=True)
    print("\n-----------------------------\n")



--- Original Prompt ---

User Prompt:
Summarize the following paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries.

Model Output:
Generative AI creates new content, such as text, images, or music, by analyzing patterns in existing data. It's commonly used in creative industries to generate original ideas and products.
-----------------------------


--- Refined Prompt ---

User Prompt:
Fasse den folgenden Absatz in genau einem Satz zusammen. 
Konzentriere dich dabei nur auf die Anwendungen in der Industrie:

Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries.

Model Output:
Generative AI wird in der 

#### [Optional] Task 4: Structured Prompting with Roles (Pirate Theme)

**Objective:**

Learn how to use structured prompts that combine role assignment, clear instructions, and examples to improve the output of language models. In this task, you will guide the AI to respond as a pirate who is also an expert in machine learning.

**Instructions:**

- Role Assignment: In your prompt, specify the role of the AI as a Machine Learning Expert who speaks like a pirate.

- Instruction: Clearly state what you want the AI to explain or discuss in pirate language.

- Examples: Provide examples to guide the AI in using pirate lingo while explaining technical concepts.

In [ ]:
# Combined Techniques Prompt with Pirate Theme

structured_prompt = """
"""

# Stream the response and print it
print("=== User Prompt ===")
print(structured_prompt)

stream = ollama.chat(
    model=model,
    messages=[{"role": "user", "content": structured_prompt}],
    stream=True,
)

print("\n=== Model Output ===")
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
print("\n")
